## Comparing Movie Recommendation Ranking methods

Here we experiment whether ranking movies based on difference between a user's rating habits/predicted ratings and a movie's average rating. We question whether the relationship between these two variables will produce more personalized reccomendation over ranking movies by their predicted ratings.

So here we compare: Standard Ranking and Difference-based ranking where Standard ranking ranks movies from their highest to lowest predicted rating and Difference-based ranking is where movies are ranked by the user's rating habits/predicted ratings compares to the movie's average rating. Then we examine the number of ratings ech recommended movie has recieved, this is to determine which method favors more popular movies.


In [4]:

import pandas as pd
from pathlib import Path
from surprise import Dataset, Reader, SVD

# Load the Letterboxd dataset
path = Path("../reviews_filtered.csv")

if not path.exists():
    path = Path("reviews_filtered.csv")

data = pd.read_csv(path)

print("Total ratings:", len(data))
print("Unique movies:", data["movie_id"].nunique())
print("Unique users:", data["user_id"].nunique())

# Load ratings into Surprise
reader = Reader(rating_scale=(1, 10))

dataset = Dataset.load_from_df(
    data[["user_id", "movie_id", "rating"]],
    reader
)

# Train SVD model
model = SVD(random_state=42)
model.fit(dataset.build_full_trainset())


Total ratings: 855434
Unique movies: 5937
Unique users: 2516


In [5]:

# Select an existing user rating history
user_counts = data["user_id"].value_counts()
user_id = user_counts.index[min(100, len(user_counts) - 1)]

# Find movies user has already rated
rated_movies = set(
    data.loc[data["user_id"] == user_id, "movie_id"]
)

# Calculate average ratings and rating counts for each movie
movie_stats = (
    data.groupby("movie_id")
    .agg(
        name=("name", "first"),
        avg_rating=("rating", "mean"),
        rating_count=("rating", "size"))
    .reset_index()
)

# Keep movies user hasn't rated and discard already rated movies
candidates = movie_stats[ ~movie_stats["movie_id"].isin(rated_movies)].copy()

# Predict the user's rating for each unrated movie
candidates["predicted_rating"] = [
    model.predict(user_id, movie_id).est
    for movie_id in candidates["movie_id"]
]

# Calculate the difference between predicted and average rating
candidates["difference"] = (
    candidates["predicted_rating"] - candidates["avg_rating"]
)

#showing some comparison
print("Selected user:", user_id)
print("Movies already rated:", len(rated_movies))
print("Movies available for recommendations:", len(candidates))

display(candidates.head(10))


Selected user: 499263
Movies already rated: 1203
Movies available for recommendations: 4734


,movie_id,name,avg_rating,rating_count,predicted_rating,difference
0,247,take-shelter,7.918605,86,7.836554,-0.082050
4,281,47-ronin-2013,4.029412,34,3.501098,-0.528313
5,282,extremely-loud-incredibly-close,4.763158,38,2.438332,-2.324826
6,285,the-great-gatsby-2013,6.092937,269,4.291889,-1.801048
7,331,total-recall-2012,4.717391,46,3.167382,-1.550009
8,603,the-muppets,7.353846,130,5.743165,-1.610681
9,994,jackass-35,7.108108,37,6.424899,-0.683209
10,1081,happy-feet-two,4.341463,41,4.528619,0.187155
13,1610,lemonade-mouth,7.333333,78,4.819393,-2.513940
14,1739,the-woman-in-black-2012,5.541176,85,4.606222,-0.934954


Process in the cell:
1. select a user that has a good history in rating movies
2. remove all the movies they already rated
3. use the SVD model we trained earler to predict how a user would rate the remaining movies
4. Calculate the difference between movies average rating from their predicted rating.

In [6]:

# Keep movies the user is predicted to enjoy
filtered = candidates[candidates["predicted_rating"] >= 7]

# Standard recommendation for user
standard_top10 = filtered.nlargest(10, "predicted_rating")

# Difference-based recommendation for user
difference_top10 = filtered.nlargest(10, "difference")

columns = [
    "name", "predicted_rating",
    "avg_rating", "difference", "rating_count"
]

# display them
print("STANDARD TOP 10:")
display(standard_top10[columns].round(2))

print("DIFFERENCE-BASED TOP 10:")
display(difference_top10[columns].round(2))

# Check how many movies are shared
overlap = len(
    set(standard_top10["movie_id"]) &
    set(difference_top10["movie_id"])
)

print("Movies shared by both lists:", overlap)


STANDARD TOP 10:


,name,predicted_rating,avg_rating,difference,rating_count
379,come-and-see,10.00,9.38,0.62,106
1564,twin-peaks-fire-walk-with-me,10.00,8.62,1.38,133
3791,twin-peaks-the-return,10.00,9.48,0.53,160
5321,the-beast-2023-1,10.00,7.83,2.17,328
1712,mulholland-drive,9.81,8.88,0.93,233
2553,inherent-vice,9.76,7.61,2.14,226
484,neon-genesis-evangelion-the-end-of-evangelion,9.75,8.82,0.93,68
1903,sunrise-a-song-of-two-humans,9.69,8.57,1.12,42
2249,lawrence-of-arabia,9.61,8.57,1.04,76
29,citizen-kane,9.58,8.50,1.08,144


DIFFERENCE-BASED TOP 10:


,name,predicted_rating,avg_rating,difference,rating_count
5321,the-beast-2023-1,10.00,7.83,2.17,328
2553,inherent-vice,9.76,7.61,2.14,226
150,cosmopolis,8.20,6.33,1.87,75
2601,blackhat,7.95,6.48,1.47,123
2566,they-came-together,7.98,6.57,1.41,69
4476,the-northman,9.15,7.77,1.38,1023
1564,twin-peaks-fire-walk-with-me,10.00,8.62,1.38,133
1576,fear-and-loathing-in-las-vegas,8.70,7.33,1.37,97
2373,under-the-skin-2013,8.73,7.42,1.30,257
1665,sex-lies-and-videotape,9.30,8.00,1.30,64


Movies shared by both lists: 3
